In [ ]:
import condor
import numpy as np

import scipy.fft as fft
import matplotlib.pyplot as plt

import os

import scipy
import scipy.interpolate as interpolate
import scipy.constants as constants

import h5py

from helper_functions import (electron_density_to_dn, det_dist_solver, write_text)

# Elementary constants
pi = constants.pi
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

In [ ]:
phot_eV = 9000 # beam energy in eV
phot_J = phot_eV * e # beam energy in J
phot_m = (h * c) / phot_J # beam wavelength in m
pulse_energy = 200e-6 # pulse energy in J
beam_pol = 'ignore' # beam polarization
beam_profile = 'gaussian' # beam profile

# Detector parameters
dsf = 4
pixel_size = dsf * 200e-6
dimY = 375 # for 4x it is 375 for 0.5 m or 369 for 0.4 m distance or 357 for 0.3 m distance
dimX = dimY

det_dist = 0.5
write_text(f'Detector distance Condor: {det_dist} m\n')
#det_dist = 0.48306051878575307 # for 0.5 m simulation
#det_dist = 0.3793928613638402 # for 0.4 m simulation
#det_dist = 0.2740333391875068 # for 0.3 m simulation
det_dist = det_dist_solver(det_dist, dimX, pixel_size)
write_text(f'Detector distance after correction: {det_dist} m\n')

cX, cY = dimX // 2, dimY // 2
pixel_num_x = dimX - dimX // 2
pixel_num_y = dimY - dimY // 2
D_particle = 15e-9 # GroEL size

# DENSS model
map3d, dx = condor.utils.emdio.read_map('denss/1ss8_denss.mrc')
map3d_scaled = electron_density_to_dn(map3d, phot_m)

# Beam diameter
focus_diam = 14e-9 # (14 nm by 14 nm)
focus_rad = focus_diam / 2
print(f'Beam diameter: {focus_diam * 1e6} um')

# Beam area 
beam_area = pi * (focus_rad ** 2)
print(f'Beam area: {beam_area * 1e12} um\u00b2') 

# Beam photon count 
beam_phots = pulse_energy / phot_J
print(f'Photon count: {beam_phots} photons')

# Beam fluence 
beam_fluence = beam_phots / beam_area
print(f'Beam fluence: {beam_fluence * 1e-12} ph/um\u00b2')
print(f'Beam fluence: {beam_fluence * phot_J * 1e-6} uJ/um\u00b2') 

# Edge resolution 
theta_pixel_x = 0.5 * np.arctan((pixel_num_x * pixel_size) / det_dist)
theta_pixel_y = 0.5 * np.arctan((pixel_num_y * pixel_size) / det_dist)

resolution_x = phot_m / (2.0 * np.sin(theta_pixel_x))
resolution_y = phot_m / (2.0 * np.sin(theta_pixel_y))
pix_real_x = 0.5 * resolution_x
pix_real_y = 0.5 * resolution_y

# Maximum corner resolution 
center_to_corner = np.sqrt((pixel_num_x * pixel_size) ** 2 + (pixel_num_x * pixel_size) ** 2)
theta_max = 0.5 * np.arctan(center_to_corner / det_dist)
resolution_max = phot_m / (2.0 * np.sin(theta_max)) 

oversampling = (det_dist * phot_m) / (pixel_size * D_particle)

shannon_groel = 1 / (D_particle * 1e9)
print(f'Oversampling ratio: {oversampling}') 
print(f'Size of Shannon pixel GroEL: {shannon_groel} nm^-1')
print(f'The maximum corner resolution is: {resolution_max * 1e9} nm') 
print(f'The resolution along X-dimension is: {resolution_x * 1e9} nm') 
print(f'The resolution along Y-dimension is: {resolution_y * 1e9} nm') 
print(f'The pixel size in real space along X-dimension is: {pix_real_x * 1e9} nm') 
print(f'The pixel size in real space along Y-dimension is: {pix_real_y * 1e9} nm') 

# Source 
source = condor.Source(wavelength=phot_m, pulse_energy=pulse_energy, focus_diameter=focus_diam,
                       polarization=beam_pol, profile_model=beam_profile)

# Particle
part_map = condor.ParticleMap(geometry='custom', map3d=map3d_scaled,
                              rotation_formalism=None, dx=dx)

particle_set = {"particle_map" : part_map}

# Detector instance 
detector = condor.Detector(distance=det_dist, pixel_size=pixel_size, nx=dimX, ny=dimY, 
                           solid_angle_correction=False)

# Experiment 
condor_experiment = condor.Experiment(source, particle_set , detector)

In [ ]:
# Simulating 3D Fourier volume of GroEL
write_text('3D propagation started!\n')
res3d = condor_experiment.propagate3d()
write_text('3D propagation finished!\n')

In [ ]:
# Casting Condor results to float64
def_rng = np.random.default_rng()
intensity3d = res3d['entry_1']['data_1']['data'].astype(np.float64)
intensity3d = intensity3d[:-1,:-1,:-1]
write_text(f'Shape of cropped intensity3d: {intensity3d.shape}')

In [ ]:
center = intensity3d.shape[0] // 2
scaling = 0.75
plt.figure(1, dpi=100)
plt.imshow(intensity3d[center,:,:]**scaling, vmin=0, vmax=0.1)
plt.title(f'<photons/pixel> ~ {intensity3d.mean(axis=(0,1,2))}', weight='bold')
plt.xticks([])
plt.yticks([])
plt.colorbar()

plt.figure(2, dpi=100)
plt.imshow(intensity3d[:,center,:]**scaling, vmin=0, vmax=0.1)
plt.xticks([])
plt.yticks([])

plt.figure(3, dpi=100)
plt.imshow(intensity3d[:,:,center]**scaling, vmin=0, vmax=0.1)
plt.xticks([])
plt.yticks([]);

In [ ]:
# Saving 3D Fourier volume with some experimental parameters to h5 file 
file_name = f'3d_groel_agipd_9_kev_denss_dsf_{dsf}x'
with h5py.File(f'{file_name}.h5','a') as f_ptr:
    f_ptr['/3Dparticle/intensity'] = intensity3d
    f_ptr['/3Dparticle/quat'] = res3d['particles']['particle_00']['extrinsic_quaternion']
    f_ptr['/exp/phot_eV'] = phot_eV
    f_ptr['/exp/pulse_energy'] = pulse_energy
    f_ptr['/exp/beam_phots'] = beam_phots
    f_ptr['/exp/pixel_size_m'] = pixel_size
    f_ptr['/exp/oversampling'] = oversampling
    f_ptr['/exp/det_dist_m'] = det_dist
    f_ptr['/exp/focus_diam_m'] = focus_diam
    f_ptr['/exp/polarization'] = beam_pol
    f_ptr['/exp/profile'] = beam_profile